In [ ]:
import pandas as pd
from slrrag.settings import OUTPUT_CSV_AGENTS_PATH, OUTPUT_CSV_FILTERED_TOPIC_PAPERS
from slrrag.input import FIRST_ROUND_QUESTIONS, OLLAMA_MODELS

df_res = pd.read_csv(OUTPUT_CSV_AGENTS_PATH)
df_res

In [ ]:
df_res.dtypes

In [ ]:
type_list_col = {
    q + "_" + x: "bool" for x in OLLAMA_MODELS for q in FIRST_ROUND_QUESTIONS.keys()
}
type_list_col

In [ ]:
df_res[type_list_col.keys()].where(
    df_res[type_list_col.keys()].isin(["True", "False", True, False]), pd.NA
)

In [ ]:
df_res_fixed = df_res.copy()
# Remove non_boolean values
df_res_fixed[list(type_list_col.keys())] = (
    df_res_fixed[type_list_col.keys()]
    .where(
        df_res_fixed[type_list_col.keys()].isin(["True", "False", True, False]), pd.NA
    )
    .replace({"True": True, "False": False})
)
df_res_fixed = df_res_fixed.astype(dtype=type_list_col)
df_res_fixed

In [ ]:
import pandas as pd
import plotly.express as px

# Reshape and count
melted_df = df_res_fixed[list(type_list_col.keys())].melt(
    var_name="question_model", value_name="is_true"
)
melted_df[["question_part", "model"]] = melted_df["question_model"].str.rsplit(
    "_", n=1, expand=True
)

# Extract the full question name (everything except the last part after the last "_")
melted_df["question"] = melted_df["question_model"].apply(
    lambda x: "_".join(x.split("_")[:-1])
)

# Count True values
true_counts = (
    melted_df[melted_df["is_true"]]
    .groupby(["question", "model"])
    .size()
    .reset_index(name="count")
)

# Plot
fig = px.bar(
    true_counts,
    x="question",
    y="count",
    color="model",
    title="Count of True Values by Question and Model",
    labels={"count": "Number of True Values", "question": "Question"},
    barmode="group",  # or "stack"
)

fig.show()

In [ ]:
df_papers = pd.read_csv(OUTPUT_CSV_FILTERED_TOPIC_PAPERS)
df_papers

In [ ]:
df_papers = df_papers.drop_duplicates("title", keep="first")
df_res_fixed = df_res_fixed.drop_duplicates("title", keep="first")
df_papers.groupby("title").size().sort_values(ascending=False)

In [ ]:
df_res_fixed = df_res_fixed.drop_duplicates("title", keep="first")
df_res_fixed.groupby("title").size().sort_values(ascending=False)

In [ ]:
df_papers[~df_papers["title"].isin(df_res_fixed["title"])]

In [ ]:
# Filter papers that are not in final results
df_res_fixed_to_merge = df_res_fixed[df_res_fixed["title"].isin(df_papers["title"])]
df_to_save = df_papers.merge(
    df_res_fixed_to_merge[list(type_list_col.keys()) + ["title"]],
    on="title",
    how="outer",
)
df_to_save

In [ ]:
from slrrag.settings import OUTPUT_CSV_TO_MANUAL_FILTER


df_to_save.to_csv(OUTPUT_CSV_TO_MANUAL_FILTER, index=False)